In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

os.makedirs("random_forest", exist_ok=True)

In [ ]:
CLASSIFIED_DIR = "clean_crop_contribution_data/aggregated_data/kmeans_classes"
FEATURES_FILE  = "engineered_climate_features_correlation_analyzed.csv"
MIN_ROWS       = 200

csv_files = glob.glob(os.path.join(CLASSIFIED_DIR, "*_classified.csv"))
print(f"Found {len(csv_files)} classified files")

Found 54 classified files


In [3]:
features_df = pd.read_csv(FEATURES_FILE)

In [4]:
all_crop_summary = []
N_SPLITS = 5

for csv_path in csv_files:
    crop_name = os.path.basename(csv_path).replace("_classified.csv", "")
    print(f"\n{'='*60}")
    print(f"CROP: {crop_name}")
    print(f"{'='*60}")

    # -- Per-crop output dir
    crop_dir = os.path.join("random_forest", f"{crop_name}_random_forest")
    os.makedirs(crop_dir, exist_ok=True)

    # -- Load & gate on row count
    class_df = pd.read_csv(csv_path)
    if len(class_df) <= MIN_ROWS:
        print(f"  Skipping - only {len(class_df)} rows (need > {MIN_ROWS})")
        continue

    # -- Merge with climate features
    class_df["location"] = class_df["State"] + "_" + class_df["District"]
    merged_df = class_df.merge(features_df, on="location", how="inner")
    print(f"  Merged shape: {merged_df.shape}")

    # -- Drop classes with too few samples to survive CV
    class_counts_raw = merged_df["Yield_Class"].value_counts()
    valid_classes    = class_counts_raw[class_counts_raw >= N_SPLITS].index.tolist()
    dropped_classes  = class_counts_raw[class_counts_raw <  N_SPLITS].index.tolist()

    if dropped_classes:
        print(f"\n  !!! Dropping classes with < {N_SPLITS} samples: {dropped_classes}")
        merged_df = merged_df[merged_df["Yield_Class"].isin(valid_classes)].copy()
        print(f"  Rows after dropping: {len(merged_df)}")

    if len(merged_df) <= MIN_ROWS:
        print(f"  Skipping - too few rows after class drop ({len(merged_df)})")
        continue

    if len(valid_classes) < 2:
        print(f"  Skipping - fewer than 2 valid classes remain")
        continue

    # -- Class counts & proportions
    class_counts = merged_df["Yield_Class"].value_counts()
    class_props  = merged_df["Yield_Class"].value_counts(normalize=True)
    print("\n  Class counts:\n", class_counts.to_string())
    print("\n  Class proportions:\n", class_props.round(3).to_string())

    # -- Encode target
    le = LabelEncoder()
    merged_df["Yield_Class_Encoded"] = le.fit_transform(merged_df["Yield_Class"])
    encoding_map = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\n  Encoding: {encoding_map}")

    # -- Build X, y
    drop_cols = ["State", "District", "location", "Median", "Max", "Yield_Class"]
    X = merged_df.drop(columns=drop_cols + ["Yield_Class_Encoded"])
    y = merged_df["Yield_Class_Encoded"]

    y           = pd.Series(LabelEncoder().fit_transform(y), index=y.index)
    n_classes   = len(np.unique(y))
    class_names = le.classes_
    print(f"  Unique classes in y after re-encode: {np.unique(y).tolist()}")

    # -- Sample weights
    classes           = np.unique(y)
    weights           = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    class_weight_dict = dict(zip(classes, weights))
    sample_weights    = y.map(class_weight_dict)

    # -- Cross-validation
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_metrics         = []
    fold_cms             = []
    fold_class_reports   = []
    shap_importance_list = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"\n  ---- Fold {fold+1} ----")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_train        = sample_weights.iloc[train_idx]

        model = RandomForestClassifier(
            n_estimators=400,
            max_depth=5,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(X_train, y_train, sample_weight=w_train)

        preds = model.predict(X_val)
        acc   = accuracy_score(y_val, preds)
        f1    = f1_score(y_val, preds, average="weighted")
        fold_metrics.append((acc, f1))
        print(f"    Accuracy: {acc:.4f}  |  F1: {f1:.4f}")

        # -- Per-class precision / recall / F1
        report_dict = classification_report(
            y_val, preds,
            labels=list(range(n_classes)),
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
        fold_class_reports.append(report_dict)
        print(f"    {'Class':<12} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
        for cls in class_names:
            r = report_dict[cls]
            print(f"    {cls:<12} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1-score']:>8.3f} {int(r['support']):>9}")

        # -- Confusion matrix
        cm = confusion_matrix(y_val, preds, labels=list(range(n_classes)))
        fold_cms.append(cm)

        # -- SHAP
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_val)

        if isinstance(shap_values, list):
            shap_vals = np.mean([np.abs(sv) for sv in shap_values], axis=0)
        elif shap_values.ndim == 3:
            shap_vals = np.abs(shap_values).mean(axis=2)
        else:
            shap_vals = np.abs(shap_values)

        shap_importance_list.append(shap_vals.mean(axis=0))

    # -- Aggregate CV metrics
    fold_metrics = np.array(fold_metrics)
    acc_mean, acc_std = fold_metrics[:, 0].mean(), fold_metrics[:, 0].std()
    f1_mean,  f1_std  = fold_metrics[:, 1].mean(), fold_metrics[:, 1].std()

    print(f"\n  CV Accuracy : {acc_mean:.4f}, SD: {acc_std:.4f}")
    print(f"  CV F1       : {f1_mean:.4f}, SD: {f1_std:.4f}")

    # -- Aggregate confusion matrix (sum all folds)
    total_cm      = np.sum(fold_cms, axis=0)
    cm_normalized = total_cm.astype(float) / total_cm.sum(axis=1, keepdims=True)

    # -- Aggregate per-class metrics across folds
    per_class_agg = {cls: {"precision": [], "recall": [], "f1-score": []}
                     for cls in class_names}
    for rd in fold_class_reports:
        for cls in class_names:
            for metric in ["precision", "recall", "f1-score"]:
                per_class_agg[cls][metric].append(rd[cls][metric])

    per_class_summary = {
        cls: {m: (np.mean(vals), np.std(vals)) for m, vals in metrics.items()}
        for cls, metrics in per_class_agg.items()
    }

    print(f"\n  Per-class CV summary (mean +/- SD across {N_SPLITS} folds):")
    print(f"    {'Class':<12} {'Precision':>16} {'Recall':>16} {'F1':>16}")
    for cls in class_names:
        p_m, p_s = per_class_summary[cls]["precision"]
        r_m, r_s = per_class_summary[cls]["recall"]
        f_m, f_s = per_class_summary[cls]["f1-score"]
        print(f"    {cls:<12} {p_m:.3f} +/- {p_s:.3f}   {r_m:.3f} +/- {r_s:.3f}   {f_m:.3f} +/- {f_s:.3f}")

    # -- SHAP importance
    shap_importance = np.mean(shap_importance_list, axis=0)
    if shap_importance.ndim > 1:
        shap_importance = np.abs(shap_importance).mean(
            axis=tuple(range(shap_importance.ndim - 1))
        )

    importance_df = pd.DataFrame({
        "feature":    X.columns,
        "importance": shap_importance
    }).sort_values("importance", ascending=False)

    # -- Save txt report
    txt_path = os.path.join(crop_dir, f"{crop_name}_results_random_forest.txt")
    with open(txt_path, "w") as f:
        f.write(f"CROP: {crop_name}\n")
        f.write(f"Total rows after merge: {merged_df.shape[0]}\n\n")

        if dropped_classes:
            f.write(f"DROPPED CLASSES (< {N_SPLITS} samples): {dropped_classes}\n\n")

        f.write("CLASS COUNTS\n")
        f.write(class_counts.to_string() + "\n\n")
        f.write("CLASS PROPORTIONS\n")
        f.write(class_props.round(4).to_string() + "\n\n")

        f.write("ENCODING\n")
        f.write(str(encoding_map) + "\n\n")

        f.write("FOLD-WISE METRICS\n")
        for i, (a, fi) in enumerate(fold_metrics, 1):
            f.write(f"  Fold {i}: Accuracy={a:.4f}  F1={fi:.4f}\n")
        f.write("\n")

        f.write("AGGREGATED CV METRICS\n")
        f.write(f"  Accuracy : {acc_mean:.4f}, SD: {acc_std:.4f}\n")
        f.write(f"  F1       : {f1_mean:.4f}, SD: {f1_std:.4f}\n\n")

        f.write("PER-CLASS CV METRICS (mean +/- SD across folds)\n")
        f.write(f"  {'Class':<12} {'Precision':>16} {'Recall':>16} {'F1':>16}\n")
        for cls in class_names:
            p_m, p_s = per_class_summary[cls]["precision"]
            r_m, r_s = per_class_summary[cls]["recall"]
            f_m, f_s = per_class_summary[cls]["f1-score"]
            f.write(f"  {cls:<12} {p_m:.3f} +/- {p_s:.3f}   {r_m:.3f} +/- {r_s:.3f}   {f_m:.3f} +/- {f_s:.3f}\n")
        f.write("\n")

        f.write("CONFUSION MATRIX (summed over all folds - rows=True, cols=Predicted)\n")
        header = "             " + "  ".join(f"{c:>10}" for c in class_names)
        f.write(header + "\n")
        for i, cls in enumerate(class_names):
            row = f"  {cls:<12}" + "  ".join(f"{total_cm[i, j]:>10d}" for j in range(n_classes))
            f.write(row + "\n")
        f.write("\n")

        f.write("CONFUSION MATRIX (row-normalized - recall per class)\n")
        f.write(header + "\n")
        for i, cls in enumerate(class_names):
            row = f"  {cls:<12}" + "  ".join(f"{cm_normalized[i, j]:>10.3f}" for j in range(n_classes))
            f.write(row + "\n")
        f.write("\n")

        f.write("SHAP FEATURE IMPORTANCE (sorted)\n")
        f.write(importance_df.to_string(index=False) + "\n")

    print(f"  Report saved -> {txt_path}")

    # -- SHAP bar plot
    fig, ax = plt.subplots(figsize=(8, 10))
    importance_df.head(20).plot(
        x="feature", y="importance", kind="barh", ax=ax, legend=False
    )
    ax.invert_yaxis()
    ax.set_title(f"{crop_name} - Top 20 SHAP Features")
    ax.set_xlabel("Mean |SHAP value|")
    plt.tight_layout()
    shap_path = os.path.join(crop_dir, f"{crop_name}_shap_random_forest.png")
    plt.savefig(shap_path, dpi=150)
    plt.close()
    print(f"  SHAP plot saved -> {shap_path}")

    # -- SHAP importance csv
    importance_df.to_csv(os.path.join(crop_dir, f"{crop_name}_shap_importance_random_forest.csv"), index=False)

    # -- Confusion matrix heatmap
    annot_labels = np.array([
        [f"{cm_normalized[i,j]:.2f}\n({total_cm[i,j]})"
         for j in range(n_classes)]
        for i in range(n_classes)
    ])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm_normalized,
        annot=annot_labels, fmt="", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        vmin=0, vmax=1, linewidths=0.5, ax=ax,
        cbar_kws={"label": "Recall (row-normalized)"}
    )
    ax.set_title(
        f"{crop_name} - Confusion Matrix\n"
        f"(all {N_SPLITS} folds summed, row-normalized | "
        f"CV Acc: {acc_mean:.3f} +/- {acc_std:.3f})"
    )
    ax.set_ylabel("True Class")
    ax.set_xlabel("Predicted Class")
    plt.tight_layout()
    cm_path = os.path.join(crop_dir, f"{crop_name}_confusion_matrix_random_forest.png")
    plt.savefig(cm_path, dpi=150)
    plt.close()
    print(f"  Confusion matrix saved -> {cm_path}")

    # -- Per-class F1 across folds
    fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 4), sharey=False)
    if n_classes == 1:
        axes = [axes]

    for ax_i, cls in enumerate(class_names):
        fold_f1s  = per_class_agg[cls]["f1-score"]
        mean_f1   = np.mean(fold_f1s)
        std_f1    = np.std(fold_f1s)
        fold_nums = range(1, N_SPLITS + 1)

        axes[ax_i].plot(fold_nums, fold_f1s, marker="o", color="darkorange", linewidth=2)
        axes[ax_i].axhline(mean_f1, color="gray", linestyle="--", label=f"mean={mean_f1:.3f}")
        axes[ax_i].fill_between(
            fold_nums, mean_f1 - std_f1, mean_f1 + std_f1,
            alpha=0.2, color="darkorange", label=f"+/-1 SD ({std_f1:.3f})"
        )
        axes[ax_i].set_title(f"Class: {cls}", fontsize=11)
        axes[ax_i].set_xlabel("Fold")
        axes[ax_i].set_ylabel("F1 Score")
        axes[ax_i].set_ylim(0, 1)
        axes[ax_i].set_xticks(list(fold_nums))
        axes[ax_i].legend(fontsize=8)

    fig.suptitle(f"{crop_name} - Per-class F1 across folds", fontsize=13, fontweight="bold")
    plt.tight_layout()
    perclass_path = os.path.join(crop_dir, f"{crop_name}_perclass_f1_random_forest.png")
    plt.savefig(perclass_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Per-class F1 plot saved -> {perclass_path}")

    # -- Collect for cross-crop summary
    row = {
        "crop":      crop_name,
        "n_rows":    merged_df.shape[0],
        "n_classes": n_classes,
        "dropped":   ", ".join(dropped_classes) if dropped_classes else "none",
        "acc_mean":  acc_mean,
        "acc_std":   acc_std,
        "f1_mean":   f1_mean,
        "f1_std":    f1_std,
    }
    for cls in class_names:
        f_m, f_s = per_class_summary[cls]["f1-score"]
        safe_cls = cls.lower().replace(" ", "_")
        row[f"f1_{safe_cls}_mean"] = round(f_m, 4)
        row[f"f1_{safe_cls}_std"]  = round(f_s, 4)

    all_crop_summary.append(row)


CROP: arecanut
  Skipping - only 152 rows (need > 200)

CROP: arhar_tur
  Merged shape: (665, 42)

  Class counts:
 Yield_Class
Medium    293
High      275
Low        97

  Class proportions:
 Yield_Class
Medium    0.441
High      0.414
Low       0.146

  Encoding: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
  Unique classes in y after re-encode: [0, 1, 2]

  ---- Fold 1 ----
    Accuracy: 0.5714  |  F1: 0.5854
    Class         Precision   Recall       F1   Support
    High              0.820    0.745    0.781        55
    Low               0.277    0.650    0.388        20
    Medium            0.611    0.379    0.468        58

  ---- Fold 2 ----
    Accuracy: 0.5263  |  F1: 0.5436
    Class         Precision   Recall       F1   Support
    High              0.829    0.618    0.708        55
    Low               0.274    0.850    0.415        20
    Medium            0.633    0.328    0.432        58

  ---- Fold 3 ----
    Accuracy: 0.5865  |  F1: 0.6110
   

In [5]:
summary_df = pd.DataFrame(all_crop_summary).sort_values("f1_mean", ascending=False)

print("\n" + "="*60)
print("CROSS-CROP SUMMARY")
print("="*60)
base_cols = ["crop", "n_rows", "n_classes", "dropped", "acc_mean", "acc_std", "f1_mean", "f1_std"]
print(summary_df[base_cols].to_string(index=False))

print("\nOverall average across all crops:")
print(f"  Accuracy : {summary_df['acc_mean'].mean():.4f}, SD: {summary_df['acc_std'].mean():.4f}")
print(f"  F1       : {summary_df['f1_mean'].mean():.4f}, SD: {summary_df['f1_std'].mean():.4f}")

summary_df.to_csv("random_forest/all_crops_summary_random_forest.csv", index=False)
print("\nSummary saved -> random_forest/all_crops_summary_random_forest.csv")


CROSS-CROP SUMMARY
                crop  n_rows  n_classes dropped  acc_mean  acc_std  f1_mean   f1_std
      other_oilseeds     223          3    none  0.879192 0.038578 0.878134 0.039197
               mesta     258          3    none  0.837330 0.035163 0.835016 0.041333
               onion     573          3    none  0.815042 0.037413 0.815197 0.036423
              garlic     439          3    none  0.810815 0.045275 0.810656 0.045382
        sweet_potato     457          3    none  0.800764 0.043204 0.804217 0.041130
        cowpea_lobia     231          3    none  0.806013 0.114713 0.803694 0.123133
                ragi     379          3    none  0.796737 0.039251 0.796649 0.039835
           sugarcane     663          3    none  0.778344 0.026687 0.792599 0.022601
           groundnut     583          2    High  0.782140 0.029276 0.785499 0.028258
      peas_and_beans     518          3    none  0.775990 0.023020 0.777663 0.023786
            turmeric     502          3    no